In [ ]:
import pandas as pd
import numpy as np
import itertools
import networkx as nx
import semopy
import itertools
import json
from warnings import filterwarnings
filterwarnings("ignore")

In [6]:
df_controle = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_controle.csv")
df_autismo = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_autismo.csv")

df_controle_prep = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_controle_prep.csv")
df_autismo_prep = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_autismo_prep.csv")

### Construção de Fórmulas de Modelos com 5 vértices conexos

In [7]:
def gerar_todas_dags_conexas_5v():
    vertices = ['Visual', 'Somatomotor', 'Default_mode', 'Cerebellar', 'Control']
    # Gera os 10 pares possíveis de ligações
    pares = list(itertools.combinations(vertices, 2))
    
    dags_validas = {}
    contador = 1
    
    # Testa todas as 59.049 combinações (3^10)
    # 0 = sem aresta, 1 = u->v, 2 = v->u
    for estados in itertools.product([0, 1, 2], repeat=10):
        G = nx.DiGraph()
        G.add_nodes_from(vertices)
        
        # Constrói o grafo com base no estado atual
        for (u, v), estado in zip(pares, estados):
            if estado == 1:
                G.add_edge(u, v)
            elif estado == 2:
                G.add_edge(v, u)
                
        # FILTRO 1: É Acíclico? (Rejeita se tiver ciclos)
        if not nx.is_directed_acyclic_graph(G):
            continue
            
        # FILTRO 2: É Conexo? (Rejeita grafos partidos ou variáveis isoladas)
        # Usamos conectividade fraca porque as setas têm direção, 
        # mas queremos apenas que a "estrutura" seja unida.
        if not nx.is_weakly_connected(G):
            continue
            
        # --- GERANDO A FÓRMULA SEM ---
        linhas_formula = []
        
        for no in vertices:
            pais = list(G.predecessors(no))
            if len(pais) > 0:
                # É endógena (recebe pelo menos uma seta)
                linhas_formula.append(f"{no} ~ {' + '.join(pais)}")
            else:
                # É exógena pura (raiz dentro deste modelo)
                linhas_formula.append(f"{no} ~~ {no}")
                
        formula = "\n".join(linhas_formula)
        arestas = [f"{u}->{v}" for u, v in G.edges()]
        num_arestas = G.number_of_edges()
        
        # Nomeia o modelo indicando o ID e o número de arestas
        nome_modelo = f"M{contador:05d}_Edges{num_arestas}"
        
        dags_validas[nome_modelo] = {
            "formula": formula,
            "Arestas": arestas,
            "Num_Arestas": num_arestas
        }
        
        contador += 1
        
    return dags_validas

# --- Executando e Verificando ---
print("Gerando modelos... Isso pode levar alguns segundos.")
modelos_conexos = gerar_todas_dags_conexas_5v()

print(f"\nTotal de DAGs Conexas geradas: {len(modelos_conexos)}")

# Exibindo um modelo não-saturado aleatório (ex: o de índice 15000)
chave_exemplo = list(modelos_conexos.keys())[15000]
print(f"\nExemplo - {chave_exemplo}:")
print(modelos_conexos[chave_exemplo]["formula"])
print("Arestas:", modelos_conexos[chave_exemplo]["Arestas"])

Gerando modelos... Isso pode levar alguns segundos.

Total de DAGs Conexas geradas: 26430

Exemplo - M15001_Edges7:
Visual ~~ Visual
Somatomotor ~ Visual
Default_mode ~ Visual
Cerebellar ~ Visual + Somatomotor + Control
Control ~ Visual + Somatomotor
Arestas: ['Visual->Somatomotor', 'Visual->Default_mode', 'Visual->Cerebellar', 'Visual->Control', 'Somatomotor->Cerebellar', 'Somatomotor->Control', 'Control->Cerebellar']


In [8]:
with open(r"..\utils\fm_modelos_5_conexo.json", "w", encoding="utf-8") as file:
    json.dump(modelos_conexos, file, indent=4)

### Ajuste de todos os modelos nos dois conjuntos.

In [9]:
def roda_os_modelos(df, modelos, retornar_inspection=False):    
    df_resultados = pd.DataFrame()
    dicionario = dict()

    for nome, modelo in modelos.items():
        formula = modelo["formula"]
        model = semopy.Model(formula)
        model.fit(df, obj="MLW")
        stats = semopy.stats.calc_stats(model)
        df_resultados = pd.concat([df_resultados, stats])
        if retornar_inspection:
            edges_inspection = model.inspect()
            dicionario[nome] = edges_inspection
            
   
    df_resultados["Modelo"] = modelos.keys()
    df_resultados.set_index("Modelo", inplace=True)

    if retornar_inspection:
        return df_resultados, dicionario

    return df_resultados


#### 1) Conjunto de Controle

##### 1.1) Dados não processados com rank e padronização.

In [10]:
df_resultados_controle = roda_os_modelos(df_controle, modelos_conexos)
df_resultados_controle

,DoF,DoF Baseline,chi2,chi2 p-value,chi2 Baseline,CFI,GFI,AGFI,NFI,TLI,RMSEA,AIC,BIC,LogLik
Modelo,,,,,,,,,,,,,,
M00001_Edges4,6,10,0.000171,1.000000e+00,1071.188557,1.005654,1.000000,1.000000,1.000000,1.009423,0.000000,17.999999,56.438895,3.234406e-07
M00002_Edges4,6,10,96.446914,0.000000e+00,1071.188558,0.914768,0.909963,0.849938,0.909963,0.857947,0.168968,17.635361,56.074257,1.823193e-01
M00003_Edges4,6,10,79.341279,4.884981e-15,1071.188555,0.930888,0.925932,0.876553,0.925932,0.884813,0.152153,17.700033,56.138929,1.499835e-01
M00004_Edges4,6,10,168.338398,0.000000e+00,1071.188641,0.847022,0.842849,0.738082,0.842849,0.745037,0.226370,17.363560,55.802456,3.182200e-01
M00005_Edges4,6,10,253.038312,0.000000e+00,1071.188558,0.767206,0.763778,0.606297,0.763778,0.612010,0.279248,17.043333,55.482229,4.783333e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
M26426_Edges9,1,10,78.853263,0.000000e+00,1071.419645,0.926652,0.926403,0.264030,0.926403,0.266518,0.383991,27.701878,87.495716,1.490610e-01
M26427_Edges10,0,10,0.034540,NaN,1071.419645,0.999967,0.999968,NaN,0.999968,NaN,inf,29.999869,94.064696,6.529242e-05
M26428_Edges9,1,10,0.026689,8.702292e-01,1071.230509,1.000917,0.999975,0.999751,0.999975,1.009172,0.000000,27.999899,87.793737,5.045158e-05


In [11]:
df_resultados_controle.to_csv(r"..\resultados\resultados_empiricos\modelos_controle.csv", index=True)

##### 1.2) Dados preprocessados com rank e padronização

In [12]:
df_resultados_controle_prep = roda_os_modelos(df_controle_prep, modelos_conexos)
df_resultados_controle_prep

,DoF,DoF Baseline,chi2,chi2 p-value,chi2 Baseline,CFI,GFI,AGFI,NFI,TLI,RMSEA,AIC,BIC,LogLik
Modelo,,,,,,,,,,,,,,
M00001_Edges4,6,10,0.000081,1.000000e+00,555.379217,1.011001,1.000000,1.000000,1.000000,1.018336,0.000000,18.000000,56.438896,1.525681e-07
M00002_Edges4,6,10,76.285016,2.087219e-14,555.379227,0.871126,0.862643,0.771072,0.862643,0.785211,0.148949,17.711588,56.150484,1.442061e-01
M00003_Edges4,6,10,50.152797,4.380886e-09,555.379227,0.919042,0.909696,0.849494,0.909696,0.865070,0.118056,17.810386,56.249282,9.480680e-02
M00004_Edges4,6,10,123.300584,0.000000e+00,555.379238,0.784919,0.777988,0.629981,0.777988,0.641532,0.192423,17.533835,55.972731,2.330824e-01
M00005_Edges4,6,10,172.463067,0.000000e+00,555.379227,0.694776,0.689468,0.482447,0.689468,0.491293,0.229227,17.347966,55.786862,3.260171e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
M26426_Edges9,1,10,45.707389,1.373035e-11,555.379248,0.918025,0.917701,0.177006,0.917701,0.180251,0.290986,27.827193,87.621031,8.640338e-02
M26427_Edges10,0,10,0.000005,NaN,555.379248,1.000000,1.000000,NaN,1.000000,NaN,inf,30.000000,94.064826,9.616710e-09
M26428_Edges9,1,10,0.000205,9.885810e-01,555.379238,1.001833,1.000000,0.999996,1.000000,1.018332,0.000000,27.999999,87.793837,3.872157e-07


In [13]:
df_resultados_controle_prep.to_csv(r"..\resultados\resultados_empiricos\modelos_controle_prep.csv", index=True)

#### 2) Conjunto autismo

##### 2.1) Dados não processados com rank e padronização.

In [14]:
df_resultados_autismo = roda_os_modelos(df_autismo, modelos_conexos)
df_resultados_autismo

,DoF,DoF Baseline,chi2,chi2 p-value,chi2 Baseline,CFI,GFI,AGFI,NFI,TLI,RMSEA,AIC,BIC,LogLik
Modelo,,,,,,,,,,,,,,
M00001_Edges4,6,10,0.000102,1.000000e+00,826.823399,1.007345,1.000000,1.000000,1.000000,1.012242,0.000000,17.999999,50.872402,3.586038e-07
M00002_Edges4,6,10,58.719813,8.188350e-11,826.823403,0.935458,0.928981,0.881636,0.928981,0.892429,0.175894,17.587931,50.460334,2.060344e-01
M00003_Edges4,6,10,73.338983,8.426593e-14,826.823412,0.917560,0.911300,0.852167,0.911300,0.862600,0.198792,17.485340,50.357743,2.573298e-01
M00004_Edges4,6,10,118.583319,0.000000e+00,826.823403,0.862169,0.856580,0.760966,0.856580,0.770282,0.257041,17.167836,50.040239,4.160818e-01
M00005_Edges4,6,10,163.558321,0.000000e+00,826.823403,0.807108,0.802185,0.670308,0.802185,0.678514,0.304079,16.852222,49.724625,5.738888e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
M26426_Edges9,1,10,55.453520,9.570122e-14,827.264008,0.933371,0.932968,0.329676,0.932968,0.333710,0.437879,27.610852,78.745701,1.945738e-01
M26427_Edges10,0,10,0.003708,NaN,827.264008,0.999995,0.999996,NaN,0.999996,NaN,inf,29.999974,84.787312,1.301156e-05
M26428_Edges9,1,10,0.005533,9.407056e-01,826.925034,1.001217,0.999993,0.999933,0.999993,1.012173,0.000000,27.999961,79.134810,1.941346e-05


In [15]:
df_resultados_autismo.to_csv(r"..\resultados\resultados_empiricos\modelos_autismo.csv", index=True)

##### 2.2) Dados preprocessados com rank e padronização

In [16]:
df_resultados_autismo_prep = roda_os_modelos(df_autismo_prep, modelos_conexos)
df_resultados_autismo_prep

,DoF,DoF Baseline,chi2,chi2 p-value,chi2 Baseline,CFI,GFI,AGFI,NFI,TLI,RMSEA,AIC,BIC,LogLik
Modelo,,,,,,,,,,,,,,
M00001_Edges4,6,10,0.000083,1.000000e+00,538.323736,1.011357,1.000000,1.000000,1.000000,1.018928,0.000000,17.999999,50.872402,2.899382e-07
M00002_Edges4,6,10,51.576855,2.268285e-09,538.323742,0.913733,0.904190,0.840316,0.904190,0.856222,0.163545,17.638057,50.510460,1.809714e-01
M00003_Edges4,6,10,54.894393,4.868770e-10,538.323742,0.907454,0.898027,0.830045,0.898027,0.845756,0.169393,17.614776,50.487179,1.926119e-01
M00004_Edges4,6,10,97.857573,0.000000e+00,538.323748,0.826134,0.818218,0.697030,0.818218,0.710223,0.232179,17.313280,50.185683,3.433599e-01
M00005_Edges4,6,10,133.727901,0.000000e+00,538.323742,0.758239,0.751585,0.585974,0.751585,0.597066,0.273784,17.061559,49.933961,4.692207e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
M26426_Edges9,1,10,37.785901,7.895032e-10,538.323753,0.930372,0.929808,0.298082,0.929808,0.303724,0.359900,27.734836,78.869684,1.325821e-01
M26427_Edges10,0,10,0.000068,NaN,538.323753,1.000000,1.000000,NaN,1.000000,NaN,inf,30.000000,84.787337,2.372279e-07
M26428_Edges9,1,10,0.000023,9.962050e-01,538.323748,1.001893,1.000000,1.000000,1.000000,1.018927,0.000000,28.000000,79.134848,7.937846e-08


In [17]:
df_resultados_autismo_prep.to_csv(r"..\resultados\resultados_empiricos\modelos_autismo_prep.csv", index=True)